In [1]:
import torch
import einops

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
m = 7
l = 9
vocab_size = 9
target = [0, 1, 2, 3, 4, 5, 8]
assert len(target) == m
target = torch.tensor(target)

In [4]:
transition_matrix = torch.zeros((l, l))

In [5]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        1.0 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),
]

In [6]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [7]:
token_probs = torch.zeros((l, vocab_size))

In [8]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.9 #prob
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.3 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    )
]

In [9]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [10]:
dp = torch.zeros((m, l))

In [11]:
dp[0, 0] = 0.9

In [12]:
# turn transition matrix, token probs, and dp into log space
transition_matrix = torch.log(transition_matrix)
token_probs = torch.log(token_probs)
dp = torch.log(dp)

In [13]:
transition_matrix.shape, token_probs.shape, dp.shape

(torch.Size([9, 9]), torch.Size([9, 9]), torch.Size([7, 9]))

In [14]:
dp[1 - 1, :].unsqueeze(0).shape

torch.Size([1, 9])

In [15]:
torch.logsumexp(dp[1 - 1, :].unsqueeze(0).T + transition_matrix, dim=0)

tensor([   -inf, -1.3093, -0.4620,    -inf,    -inf,    -inf,    -inf,    -inf,
           -inf])

In [16]:
token_probs[:, target[1]]

tensor([   -inf, -0.3567, -1.6094,    -inf,    -inf,    -inf, -2.3026,    -inf,
           -inf])

In [17]:
token_probs[:, target[1]] + (torch.logsumexp(dp[1 - 1, :].unsqueeze(0).T + transition_matrix, dim=0))

tensor([   -inf, -1.6660, -2.0715,    -inf,    -inf,    -inf,    -inf,    -inf,
           -inf])

In [18]:
# currently transition_matrix, token_probs, and dp are 2d tensors
# copy them to mimic batch size of 2
transition_matrix = transition_matrix.unsqueeze(0).repeat(2, 1, 1)
token_probs = token_probs.unsqueeze(0).repeat(2, 1, 1)
dp = dp.unsqueeze(0).repeat(2, 1, 1)

In [19]:
# for debugging purposes, perturb the token probs second batch
# token_probs[1, :, :] += 0.1

In [20]:
# do the same for the target
target = target.unsqueeze(0).repeat(2, 1)

In [21]:
target, target.shape

(tensor([[0, 1, 2, 3, 4, 5, 8],
         [0, 1, 2, 3, 4, 5, 8]]),
 torch.Size([2, 7]))

In [22]:
transition_matrix.shape, token_probs.shape, dp.shape

(torch.Size([2, 9, 9]), torch.Size([2, 9, 9]), torch.Size([2, 7, 9]))

In [23]:
dp.shape, transition_matrix.shape

(torch.Size([2, 7, 9]), torch.Size([2, 9, 9]))

In [24]:
torch.logsumexp(dp[:, 1-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1)

tensor([[   -inf, -1.3093, -0.4620,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf],
        [   -inf, -1.3093, -0.4620,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf]])

In [25]:
torch.logsumexp(dp[:, 1-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1).shape

torch.Size([2, 9])

In [26]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [37]:
def dag_loss(targets, transition_matrix, emission_probs, bos_idx=0):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.zeros((batch_size, m, l))
    bos_emissions = emission_probs[:, 0, bos_idx]
    dp[:, 0, 0] = bos_emissions
    # dp is almost setup correctly, just need to replace every 0 with -inf
    dp[dp == 0] = -float('inf')
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (torch.logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [38]:
vector_gather(token_probs.transpose(1, 2), target[:, 1]) + (torch.logsumexp(dp[:, 1-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))

tensor([[   -inf, -1.6660, -2.0715,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf],
        [   -inf, -1.6660, -2.0715,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf]])

In [39]:
dp = dag_loss(target, transition_matrix, token_probs)

In [40]:
print(dp.round(decimals=3))

tensor([[[ -0.1050,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,  -1.6660,  -2.0710,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,     -inf,  -1.8890,  -4.3740,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,     -inf,     -inf,  -1.9950,  -6.6770,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,  -2.1000,     -inf,  -9.6720,
              -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,     -inf,  -3.3040,  -5.0960,
          -11.2820,     -inf],
         [    -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
           -6.7050,  -3.6600]],

        [[ -0.1050,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,  -1.6660,  -2.0710,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],

In [41]:
# doing it by hand by enumerating all possible paths
p1m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.6, 0.7])
p1t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

# p2m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.1, 0.2])
# p2t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

p3m = torch.tensor([0.9, 0.2, 0.1, 0.1, 0.1, 0.2, 0.7])
p3t = torch.tensor([0.7, 1, 1, 0.5, 1, 1])

In [42]:
p1 = p1m.prod() * p1t.prod()
# p2 = p2m.prod() * p2t.prod()
p3 = p3m.prod() * p3t.prod()

In [43]:
# acc = p1 + p2 + p3
acc = p1 + p3

In [44]:
acc

tensor(0.0257)

In [45]:
dp

tensor([[[ -0.1054,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,  -1.6660,  -2.0715,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,     -inf,  -1.8892,  -4.3741,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,     -inf,     -inf,  -1.9945,  -6.6766,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,  -2.0999,     -inf,  -9.6724,
              -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,     -inf,  -3.3038,  -5.0956,
          -11.2818,     -inf],
         [    -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
           -6.7050,  -3.6602]],

        [[ -0.1054,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],
         [    -inf,  -1.6660,  -2.0715,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf],

In [46]:
dp = torch.exp(dp)

In [49]:
# check difference between dp[m-1, l-1] and acc
print(dp[0][m-1, l-1] - acc)
print(dp[1][m-1, l-1] - acc)

tensor(-1.0408e-17)
tensor(-1.0408e-17)


I understand the problem now! Before, the dynamic programming approach and the brute-force considering all paths approach were slightly different, and it was bothering me as to why, but I think I figured it out. 

It actually explains a lot too, but the point is, we want to ensure all paths end at the same vertex, in this case, vertex 9. Since the target has length 7, this means we want to consider all paths of length 7 that end up at vertex 9 (that is, all paths that contain exactly 6 edges and end at vertex 9).

Before in my brute force attempt, I was considering all paths of length 7, but did not ensure that they ended at vertex 9, in the variable `p2m`, I was ending at vertex 8, and thus my result was over what it should have been.